In [4]:
import numpy as np
import random
import matplotlib.pyplot as plt

from remi.system import System
from remi.visualize import plot_states, plot_controls, animate
from remi.kinematics import (calc_capture_point_position,
                             calc_end_effector_position,
                             calc_end_effector_velocity,
                             calc_capture_point_velocity,
                             calc_capture_point_acceleration,
                             calc_J,
                             calc_J_dot)
from remi.clik_functions import get_gam2_vals, calc_clik_params
from remi.dynamics import inverse_dynamics

# Placeholders à remplacer avec les vraies fonctions
def simulatePendulumCart(params, theta_des, simu_time, dt):
    # Remplacer ceci par le vrai modèle
    settling_time = np.random.uniform(0, 5)
    static_error = np.random.uniform(0, 1)
    return settling_time, static_error

def fitnesse(settling_time, static_error, flag):
    if flag == 0:
        return settling_time + 10 * static_error
    else:
        return 10 * static_error

def final_simu(description, fitness, pop):
    print(f"Final simulation ({description}): Best fitness = {min(fitness)}")
    return pop[np.argmin(fitness)]


In [ ]:
## useless cell for now, don't take in account
popsize = 50
maxgen = 50
simu_time = 10  # secondes
elitism_rate = 0.1
selection_rate = 0.5
theta_des = 0
dt = 0.01

pop = []
fitness = []
flag = 0

for _ in range(popsize):
    chr = np.zeros((15, 3))
    chr[0, 0] = -np.pi
    chr[2, 2] = np.pi
    chr[3, 0] = -10
    chr[5, 2] = 10

    angles = np.sort(2 * np.pi * np.random.rand(7) - np.pi)
    speeds = np.sort(20 * np.random.rand(7) - 10)
    params = np.vstack((np.concatenate(([-np.pi], angles, [np.pi])),
                        np.concatenate(([-10], speeds, [10]))))

    chr[0:3, :] = params[0, [0, 1, 3]], params[0, [2, 4, 6]], params[0, [5, 7, 8]]
    chr[3:6, :] = params[1, [0, 1, 3]], params[1, [2, 4, 6]], params[1, [5, 7, 8]]

    for k in range(6, 15):
        chr[k, :] = 20 * np.random.rand(3) - 10

    pop.append(chr)

def triangle(x, le, ce, re):
    """
    Triangle membership function.
    Returns the degree of membership (mu) of x in the triangular fuzzy set
    defined by points le (left), ce (center), and re (right).
    """
    if le <= x < ce:
        mu = (x - le) / (ce - le)
    elif ce <= x <= re:
        mu = (re - x) / (re - ce)
    else:
        mu = 0.0
    return mu

def fis(chr,flag,q,qdot,djk,djkdot):
    mu1=[]
    mu2=[]
    rmu=[]
    pol=[]
    if flag==0: # compute the degree of membership depending of if we want p value of w value
        for i in range(3): 
            mu1.append(triangle(q,chr[i][0],chr[i][1],chr[i][2]))
            mu2.append(triangle(qdot,chr[i+3][0],chr[i+3][1],chr[i+3][2]))
    elif flag==1 :
        mu1.append(triangle(djk,chr[i][0],chr[i][1],chr[i][2]))
        mu2.append(triangle(djkdot,chr[i+3][0],chr[i+3][1],chr[i+3][2]))


    for n in range(3): # compute the rule activation
        for m in range(3):
            rmu.append(min(mu1[n],mu2[m]))
    agregg=np.sum(rmu)

    for rule in range(9):
        pol.append((chr[rule+6][0]+chr[rule+6][1]+chr[rule+6][2])*rmu[rule])
    output=np.sum(pol)/agregg
    return output


In [58]:
pop[0][0]

array([[-1.00000000e+00,  9.43970733e-04,  5.05039054e-01],
       [ 1.94360897e-01,  8.13299746e-01,  1.07684400e+00],
       [ 9.34400019e-01,  1.17548838e+00,  1.00000000e+00],
       [-1.00000000e+00, -8.62889944e-02,  3.39399446e-01],
       [ 1.69341604e-01,  6.55798503e-01,  1.02379244e+00],
       [ 9.03104238e-01,  1.03795425e+00,  1.00000000e+00],
       [ 1.29366108e+00,  4.96463978e-01,  7.03096269e-01],
       [ 1.72426755e+00,  4.64877190e-01,  4.13097664e-01],
       [ 4.59028356e-01,  5.03333968e-01,  1.11438432e+00],
       [ 5.33962830e-01,  1.12825471e+00,  1.63456118e+00],
       [ 7.79390028e-01,  1.30987461e+00,  8.54930487e-01],
       [ 1.13492173e+00,  6.03474968e-02,  4.21446981e-01],
       [ 6.88263471e-01,  1.26800907e+00,  1.66092742e+00],
       [ 6.09924209e-01,  5.65772846e-01,  1.99386459e-01],
       [ 5.09805788e-01,  3.91800845e-01,  9.65281251e-01]])

In [59]:
q=0.2
qdot=0.3
fis(pop[0][0],0,q,qdot,0,0)

np.float64(2.60053683771935)

In [ ]:
def fitness(chr):
# getting the simulation running
# Physical Parameters
    r_s = np.array([-1.25, 0.])
    r_t = np.array([1.25, 0.])
    rho = np.array([0.5, 0.5, 0.5, 0.5])
    m = np.array([250., 25., 25., 180.])
    I = np.array([25., 2.5, 2.5, 18.])
    d = np.zeros(4)

    # Simulation Settings
    t_dur = 5.
    step_size = 0.1
    tol = 0.01
    max_tau = (np.inf, np.inf, np.inf, 0.)

    # Initial Conditions
    y0 = np.array([np.pi/3.,    # theta_s
                -0.3,        # theta_1
                -0.1,        # theta_2
                0.,          # theta_t
                0.,          # theta_dot_s
                0.,          # theta_dot_1
                0.,          # theta_dot_2
                0.2])       # theta_dot_t


    # Put parameters and settings in dict
    parameters = dict(r_s=r_s,
                    r_t=r_t,
                    rho=rho,
                    m=m,
                    I=I,
                    d=d)

    settings = dict(t_dur=t_dur,
                    step_size=step_size,
                    tol=tol,
                    max_tau=max_tau)

    # Define system
    sys = System(y0, parameters, settings)

    # CLIK control goes here...
    def controls(t, y):
        # System limits
        q_bar = np.zeros(3)
        q_max = np.array([2.*np.pi, # theta_s limit
                        np.pi/2.,   # theta_1 limit
                        np.pi])     # theta_2 limit
        q_min = -q_max

        # Specific states
        q = y[:3]
        qdot = y[4:-1]

        # Compute error and derivative error
        xd = calc_capture_point_position(y, rho, r_t)
        xe = calc_end_effector_position(y, rho, r_s)
        vd = calc_capture_point_velocity(y, rho, r_t)
        ve = calc_end_effector_velocity(y, rho, r_s)

        e = xd - xe
        e_dot = vd - ve

        # Capture point acceleration used in qddot_P calc
        ad = calc_capture_point_acceleration(y, 0., rho, r_t)

        # Calc and cache Jacobian information
        J = calc_J(y, rho, r_s)
        J_dot = calc_J_dot(y, rho, r_s)

        M = J @ J.T
        M_inv = np.linalg.pinv(M)
        M_dot = J_dot @ J.T + J @ J_dot.T
        Jp = J.T @ M_inv
        Jp_dot = (J_dot.T @ M_inv) - (J.T @ M_inv @ M_dot @ M_inv)

        # Calculate parameters for gam2 calculation and for inference input
        pos, vel, djk, djk_dot = get_gam2_vals(y, rho, r_s, r_t)
        
        # INFER P, W, KP, KD, AND KN HERE
        p=[]
        w=[]
        for pval in range(3):
            p.append(fis(chr,0,q,qdot,J,J_dot))
        for wval in range(6):
            w.append(fis(chr,1,q,qdot,djk,djk_dot))
        """ to finish, need to see the structure of djk and djkdot"""







        #p = np.array([2.225073858507201e-308, 1e-2, 1e-2])
        #w = np.ones(6)*2. # w11 w12 w21 w22 w31 w32
        KD = 5.
        KP = 5.
        KN = 1.

        # Calculate parameters for qddot_S
        lam, lam_dot = calc_clik_params(y, rho, r_s, pos, vel, p, q_bar, q_max, q_min, w)
        edot_N = (np.eye(3) - Jp@J)@(lam.squeeze() - qdot)

        # Apply gains and previous information to get primary desired
        # acceleration and secondary desired acceleration
        qddot_P = (Jp@(ad + KD*e_dot + KP*e - (J_dot@qdot)[:, None])).squeeze()
        qddot_S = (np.eye(3) - Jp@J)@(lam_dot.squeeze() + KN*edot_N) - \
                (Jp@J_dot@Jp + Jp_dot)@J@(lam.squeeze() - qdot)

        # Calculate complete desired acceleration
        qddot = qddot_P + qddot_S

        # Compute inverse dynamics to find proper control input
        tau = inverse_dynamics(y, np.hstack((qddot, 0.)), r_s, r_t, rho, m, I, d)
        return tau


    def event(t, y, tol):
        ee = calc_end_effector_position(y, rho, r_s)
        capt = calc_capture_point_position(y, rho, r_t)
        dist = np.sqrt((ee[0] - capt[0])**2 + (ee[1] - capt[1])**2)

        return dist <= tol

    # set the controls and events
    sys.set_controller(controls)
    sys.set_event(event)

    # get the results 
    sol = sys.run()



    # control effort as a fitness
    def calculate_cost(sol):
        if sol.status == -1 :
            return 1e100
        elif sol.status ==0 :
            penalty = 1e10
        else:
            penalty=0
            
        isu=np.sum(sol.u**2)*0.1 #0.1 is the time step 
        
        if not np.isfinite(isu):
            return 5e7
        
        return isu+penalty

    cost=calculate_cost(sol)
    return(cost)

In [26]:
def init(flag):
    pop=[]
    
    # Uniform distribution in [-0.5, 1.5]
    indiv=[]
    if flag==0: # to generate FIS for p values
        for p in range(2): # generate a 6*3 matrix for the triangles vertices of the 2 inputs
            new_mf=[]
            mf = np.sort(2.0 * np.random.rand(1, 9) - 0.5,axis=1) # gives a membership function distribution between -1 and 1 
            mf[0,0] = -1.
            mf[0,8] = 1.
            k=0
            new_mf.append(mf[0,[3*k,3*k+1,3*k+3]])
            for k in [1,2]:
                if k==1:
                    new_block=np.array(mf[0,[3*k-1,3*k+1,3*k+3]])
                else:
                    new_block=np.array(mf[0,[3*k-1,3*k+1,3*k+2]])
                new_mf = np.vstack([new_mf,new_block]) # generate the 3*3 membership matrix for the input
            if len(indiv) == 0:
                indiv=new_mf 
            else:
                indiv=np.vstack([indiv,new_mf])  
        indiv=np.vstack([indiv,2.0 * np.random.rand(9, 3)]) # generate the TSK parameters for the 9 rules and combine with MFs, so a 15*3
        pop.append(indiv)


    elif flag==1: # generate FIS for wjk values
        for p in range(2): # generate a 6*3 matrix for the triangles vertices of the 2 inputs
            new_mf=[]
            mf = np.sort(2.0 * np.random.rand(1, 9) - 0.5,axis=1) # gives a membership function distribution between 0 and 1
            mf[0,0] = 0
            mf[0,8] = 1.
            k=0
            new_mf.append(mf[0,[3*k,3*k+1,3*k+3]])
            for k in [1,2]:
                if k==1:
                    new_block=np.array(mf[0,[3*k-1,3*k+1,3*k+3]])
                else:
                    new_block=np.array(mf[0,[3*k-1,3*k+1,3*k+2]])
                new_mf = np.vstack([new_mf,new_block]) # generate the 3*3 membership matrix for the input
            if len(indiv) == 0:
                indiv=new_mf 
            else:
                indiv=np.vstack([indiv,new_mf])  
        indiv=np.vstack([indiv,2.0 * np.random.rand(9, 3)]) # generate the TSK parameters for the 9 rules and combine with MFs, so a 15*3
        pop.append(indiv)
    return pop


def selection(pop, fitness_vals, rate, popsize, elitism_rate):
    num_selected = round(rate * popsize)
    num_elites = int(np.ceil(elitism_rate * popsize))
    sorted_idx = np.argsort(fitness_vals)
    elites = [pop[i] for i in sorted_idx[:num_elites]]

    selected = []
    for _ in range(num_selected - num_elites):
        candidates = random.sample(range(popsize), 3)
        best = min(candidates, key=lambda idx: fitness_vals[idx])
        selected.append(pop[best])

    return num_elites, elites + selected

def mutation(child, mutation_rate): # for 1 FIS, to do for 9
    if random.random() < mutation_rate:
        for i in range(3):
            for n in range(15):
                multiplier = 1 + 0.6 * (random.random() - 0.5)
                child[n, i] *= multiplier
        # insuring full coverage of the inputs
        child[0, 0] = -1
        child[2, 2] = 1
        child[3, 0] = 0
        child[5, 2] = 1
    return child

def reproduction(pool, mutation_rate, popsize, num_elites): # for 1 FIS, to do for 9
    # interpolation of the membership functions and 1 point crossover for the rules
    new_pop = []
    for i in range(num_elites):
        new_pop.append(pool[i])
    for i in range(num_elites, popsize):
        parent1 = random.choice(pool)
        parent2 = random.choice(pool)
        child = np.zeros((15, 3))
        for k in range(6): 
            alpha = random.random()
            beta = 1 - alpha
            child[k, :] = alpha * parent1[k, :] + beta * parent2[k, :]
        child = mutation(child, mutation_rate)
        # insuring full coverage of the inputs
        child[0, 0] = -1
        child[2, 2] = 1
        child[3, 0] = 0
        child[5, 2] = 1

        # crossover for the rules
        for k in [6,7, 8, 9]:
            child[k,:]=parent1[k,:]
        for k in [10,11,12,13,14]:
            child[k,:]=parent2[k,:]

        new_pop.append(child)
    return new_pop


In [56]:
#base initialisation of every chromosome
# we have a total of 9 FIS and add 5 for the KP, KD, Kn
pop=[]
for p in range(popsize):
    pop.append(init(0))
    for i in range(1,9): # add to this for loop to add kp kd kn FIS
        if i<=2:
            pop[p]=np.vstack([pop[p],init(0)])
        else:
            pop[p]=np.vstack([pop[p],init(1)])



In [ ]:
evo = []
gen = 0
while gen < maxgen:
    mutation_rate = max(0.3, 1 - gen / maxgen)
    num_elites, pool = selection(pop, fitness, selection_rate, popsize, elitism_rate)
    pop = reproduction(pool, mutation_rate, popsize, num_elites)
    fitness = []
    for chr in pop:
        settling_time, static_error = simulatePendulumCart(chr, theta_des, simu_time, dt)
        fitness.append(fitnesse(settling_time, static_error, flag))
    evo.append(min(fitness))
    print(f"Generation {gen}, Best Fitness: {evo[-1]}")
    gen += 1
out1 = final_simu("with settling time", fitness, pop)


In [ ]:
# Nouvelle initialisation
flag = 1
pop = []
fitness = []
for _ in range(popsize):
    chr = np.zeros((15, 3))
    chr[0, 0] = -np.pi
    chr[2, 2] = np.pi
    chr[3, 0] = -10
    chr[5, 2] = 10

    angles = np.sort(2 * np.pi * np.random.rand(7) - np.pi)
    speeds = np.sort(20 * np.random.rand(7) - 10)
    params = np.vstack((np.concatenate(([-np.pi], angles, [np.pi])),
                        np.concatenate(([-10], speeds, [10]))))

    chr[0:3, :] = params[0, [0, 1, 3]], params[0, [2, 4, 6]], params[0, [5, 7, 8]]
    chr[3:6, :] = params[1, [0, 1, 3]], params[1, [2, 4, 6]], params[1, [5, 7, 8]]

    for k in range(6, 15):
        chr[k, :] = 20 * np.random.rand(3) - 10

    pop.append(chr)

for chr in pop:
    settling_time, static_error = simulatePendulumCart(chr, theta_des, simu_time, dt)
    fitness.append(fitnesse(settling_time, static_error, flag))

evo = []
gen = 0
while gen < maxgen:
    mutation_rate = max(0.1, 1 - gen / maxgen)
    num_elites, pool = selection(pop, fitness, selection_rate, popsize, elitism_rate)
    pop = reproduction(pool, mutation_rate, popsize, num_elites)
    fitness = []
    for chr in pop:
        settling_time, static_error = simulatePendulumCart(chr, theta_des, simu_time, dt)
        fitness.append(fitnesse(settling_time, static_error, flag))
    evo.append(min(fitness))
    print(f"Generation {gen}, Best Fitness: {evo[-1]}")
    gen += 1
out2 = final_simu("without settling time", fitness, pop)
